In [1]:
# ============================================================
# Quick Production Pipeline Test
# ============================================================

import joblib
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# Load Saved Production Artifacts
# ============================================================

preprocessor = joblib.load("../models/preprocessor.pkl")

best_xgb = joblib.load("../models/xgb.pkl")

best_lgb = joblib.load("../models/lightgbm.pkl")

best_mlp = joblib.load("../models/mlp.pkl")

meta_model = joblib.load("../models/meta.pkl")

calibrated_model = joblib.load("../models/calibrated.pkl")

best_threshold = joblib.load("../models/threshold.pkl")

feature_names = joblib.load("../models/feature_names.pkl")

target_labels = joblib.load("../models/target_labels.pkl")

print("✓ All artifacts loaded successfully.")

✓ All artifacts loaded successfully.


In [3]:
# ============================================================
# Load One Sample
# ============================================================

DATA_PATH = "../data/train.csv"

data = pd.read_csv(DATA_PATH)

TARGET = "diagnosed_diabetes"

sample = data.drop(columns=[TARGET, "id"]).iloc[[0]]

true_label = data[TARGET].iloc[0]

print("Sample loaded successfully.")
print(f"True Label : {true_label}")

Sample loaded successfully.
True Label : 1.0


In [4]:
# ============================================================
# Run Prediction Pipeline
# ============================================================

sample_processed = preprocessor.transform(sample)

xgb_prob = best_xgb.predict_proba(sample_processed)[:, 1]

lgb_prob = best_lgb.predict_proba(sample_processed)[:, 1]

mlp_prob = best_mlp.predict_proba(sample_processed)[:, 1]

meta_features = pd.DataFrame({

    "XGB": xgb_prob,

    "LGBM": lgb_prob,

    "MLP": mlp_prob

})

probability = calibrated_model.predict_proba(meta_features)[:, 1][0]

prediction = int(probability >= best_threshold)

print(f"Predicted Probability : {probability:.6f}")
print(f"Prediction            : {prediction}")
print(f"Class                 : {target_labels[prediction]}")

Predicted Probability : 0.477167
Prediction            : 1
Class                 : Diabetic


d:\Projects\Diabetes-Risk-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# ============================================================
# Verification
# ============================================================

print("=" * 60)

print("Production Pipeline Verification")

print("=" * 60)

print(f"Probability : {probability:.6f}")

print(f"Threshold   : {best_threshold:.6f}")

print(f"Prediction  : {target_labels[prediction]}")

print(f"True Label  : {target_labels[true_label]}")

print("=" * 60)

if 0.0 <= probability <= 1.0:
    print("✓ Probability check passed.")
else:
    print("✗ Probability check failed.")

if prediction in [0, 1]:
    print("✓ Prediction check passed.")
else:
    print("✗ Prediction check failed.")

print("\n✓ All production artifacts loaded successfully.")
print("✓ End-to-end inference completed successfully.")

Production Pipeline Verification
Probability : 0.477167
Threshold   : 0.382094
Prediction  : Diabetic
True Label  : Diabetic
✓ Probability check passed.
✓ Prediction check passed.

✓ All production artifacts loaded successfully.
✓ End-to-end inference completed successfully.
